# 07. Binary Conv–Transformer Intermediate Fusion

            ## Goal

            기존 `lstm_dataset.pkl`을 그대로 사용하여 다음 세 모델을 subject-safe 4-fold OOF로 비교합니다.

            1. 실제 최고 체크포인트 구조의 early-fusion LSTM baseline
            2. day-wise Conv-only intermediate fusion
            3. day-wise Conv + 1-block Transformer intermediate fusion

            공식 `split == "val"`은 모델, epoch, threshold를 모두 고정한 뒤 한 번만 평가합니다.
            primary metric은 subject-level ROC-AUC입니다. 이 노트북은 실행 가능한 전체 코드를 포함하며
            옆의 Python script에 의존하지 않습니다.


## Setup — Colab 경로와 실행 모드

            아래에서 바꿔야 하는 **경로 override는 `DATA_PATH_OVERRIDE` 하나뿐**입니다.
            결과는 데이터 파일 옆 `binary_conv_transformer_results/YYYYMMDD_HHMMSS/`에 생성됩니다.
            `QUICK_MODE=True`는 코드 경로 점검용이며 공식 결과로 사용하면 안 됩니다.


In [1]:

try:
    from google.colab import drive
    drive.mount("/content/drive")
except (ImportError, ModuleNotFoundError):
    print("Colab 외부 환경: Google Drive mount를 건너뜁니다.")

DATA_PATH_OVERRIDE = "/content/drive/MyDrive/GoogleAI_contest/Hyunsoo/previous/privious_lstm_preprocessed/lstm_dataset.pkl"
NORMALIZATION_MODE = "match_baseline"  # 또는 "stored_minmax"
QUICK_MODE = False


Mounted at /content/drive


## Setup — 라이브러리, seed, 실험 설정

TensorFlow/Keras Functional API, 재현성 seed, mixed precision, 모델 이름과 설정 dataclass를 정의합니다. TensorFlow가 없으면 원인이 드러나는 오류를 냅니다.


In [2]:
import argparse
import gc
import json
import os
import pickle
import random
import time
from collections import Counter
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Iterable, Sequence
from zoneinfo import ZoneInfo

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, regularizers
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "TensorFlow가 없습니다. Google Colab GPU 런타임을 사용하거나 "
        "`pip install tensorflow` 후 다시 실행하세요."
    ) from exc


SEOUL = ZoneInfo("Asia/Seoul")
PADDING_VALUE = -1.0
CLASS_MAPPING = {0: "CN", 1: "MCI+Dementia"}
MODEL_A = "A_early_fusion_lstm"
MODEL_B = "B_conv_only_intermediate"
MODEL_C = "C_conv_transformer_intermediate"
MODEL_ORDER = (MODEL_A, MODEL_B, MODEL_C)
MODEL_SHORT_NAME = {MODEL_A: "model_a", MODEL_B: "model_b", MODEL_C: "model_c"}
AGGREGATION_METHOD = "mean_window_probability"

# 실제 final_model.keras에 입력된 고정 top-20 feature다.
BASELINE_SELECTED_FEATURES = [
    "sleep_light",
    "sleep_breath_average",
    "sleep_score_deep",
    "activity_inactive",
    "activity_score_stay_active",
    "activity_met_5min_t035",
    "activity_cal_total",
    "sleep_hr_lowest",
    "sleep_rmssd",
    "sleep_restless",
    "sleep_deep",
    "activity_met_5min_t220",
    "activity_met_min_high",
    "activity_met_5min_t039",
    "activity_met_5min_t221",
    "activity_met_5min_t033",
    "activity_met_5min_t034",
    "sleep_hr_average",
    "activity_high",
    "activity_met_5min_t212",
]


@dataclass(frozen=True)
class ExperimentConfig:
    data_path: Path
    result_root: Path = Path("binary_conv_transformer_results")
    normalization_mode: str = "match_baseline"
    quick_mode: bool = False
    n_splits: int = 4
    seed: int = 42
    batch_size: int = 32
    max_epochs: int = 70
    baseline_max_epochs: int = 80
    patience: int = 10
    learning_rate: float = 3e-4
    baseline_learning_rate: float = 5e-4
    clipnorm: float = 1.0
    use_mixed_precision: bool = True
    inverse_subject_window_weight: bool = True
    subject_class_weight: bool = True
    quick_subjects_per_class: int = 8
    transformer_min_auc_gain: float = 0.01

    def validated(self) -> "ExperimentConfig":
        if self.normalization_mode not in {"stored_minmax", "match_baseline"}:
            raise ValueError(
                "NORMALIZATION_MODE은 'stored_minmax' 또는 'match_baseline'이어야 합니다: "
                f"{self.normalization_mode!r}"
            )
        if self.n_splits != 4 and not self.quick_mode:
            raise ValueError("공식 실험은 subject-level stratified 4-fold여야 합니다.")
        if not Path(self.data_path).exists():
            raise FileNotFoundError(
                f"lstm_dataset.pkl을 찾지 못했습니다: {self.data_path}\n"
                "notebook 상단 DATA_PATH_OVERRIDE 또는 --data-path를 수정하세요."
            )
        return self


@dataclass
class DatasetBundle:
    X_cont: np.ndarray
    X_disc: np.ndarray
    X_integrated: np.ndarray
    y: np.ndarray
    patient_id: np.ndarray
    split: np.ndarray
    window_start: np.ndarray
    window_end: np.ndarray
    continuous_feature_names: list[str]
    discrete_feature_names: list[str]
    integrated_feature_names: list[str]
    meta: dict[str, Any]


@dataclass
class ExperimentArrays:
    X_cont_train: np.ndarray
    X_disc_train: np.ndarray
    X_integrated_train: np.ndarray
    y_train: np.ndarray
    groups_train: np.ndarray
    start_train: np.ndarray
    end_train: np.ndarray
    X_cont_val: np.ndarray
    X_disc_val: np.ndarray
    X_integrated_val: np.ndarray
    y_val: np.ndarray
    groups_val: np.ndarray
    start_val: np.ndarray
    end_val: np.ndarray
    integrated_feature_names: list[str]


@dataclass
class FoldDefinition:
    fold: int
    train_indices: np.ndarray
    val_indices: np.ndarray
    train_subjects: list[str]
    val_subjects: list[str]


def set_global_seed(seed: int) -> None:
    """Python, NumPy, TensorFlow의 난수 seed를 함께 고정한다."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def configure_tensorflow(config: ExperimentConfig) -> dict[str, Any]:
    gpus = tf.config.list_physical_devices("GPU")
    if gpus and config.use_mixed_precision:
        keras.mixed_precision.set_global_policy("mixed_float16")
    else:
        keras.mixed_precision.set_global_policy("float32")
    set_global_seed(config.seed)
    info = {
        "tensorflow_version": tf.__version__,
        "keras_version": getattr(keras, "__version__", "unknown"),
        "gpu_devices": [device.name for device in gpus],
        "mixed_precision_policy": keras.mixed_precision.global_policy().name,
    }
    print("TensorFlow 설정:", json.dumps(info, ensure_ascii=False))
    return info


def json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


## Checks — 기존 pickle 로드와 데이터 계약

새 전처리 파일을 만들지 않고 기존 pickle을 읽습니다. shape, 라벨, NaN/Inf, train/val subject 중복, 환자 라벨 충돌, window 중복과 변수별 padding 비율을 확인합니다.


In [3]:
def _numpy_frombuffer_compat(
    buffer: bytes,
    dtype: np.dtype,
    shape: Sequence[int],
    order: str,
    axis_order: Sequence[int] | None = None,
) -> np.ndarray:
    array = np.frombuffer(buffer, dtype=dtype)
    if order == "K" and axis_order is not None:
        return array.reshape(shape, order="C").transpose(axis_order)
    return array.reshape(shape, order=order)


class CompatibleNumpyUnpickler(pickle.Unpickler):
    """NumPy 1/2 계열 간 `_frombuffer` pickle 경로 차이를 흡수한다."""

    def find_class(self, module: str, name: str) -> Any:
        if module == "numpy._core.numeric" and name == "_frombuffer":
            return _numpy_frombuffer_compat
        return super().find_class(module, name)


def load_dataset(path: Path) -> DatasetBundle:
    required_keys = {
        "X_continuous_seq",
        "X_discrete_seq",
        "X_integrated_seq",
        "y",
        "patient_id",
        "split",
        "window_start_date",
        "window_end_date",
    }
    print(f"pickle 로드: {path}")
    with Path(path).open("rb") as file:
        data = CompatibleNumpyUnpickler(file).load()
    missing = sorted(required_keys - set(data))
    if missing:
        raise KeyError(f"pickle 필수 key 누락: {missing}; 발견 key={sorted(data)}")

    bundle = DatasetBundle(
        X_cont=np.asarray(data["X_continuous_seq"], dtype=np.float32),
        X_disc=np.asarray(data["X_discrete_seq"], dtype=np.float32),
        X_integrated=np.asarray(data["X_integrated_seq"], dtype=np.float32),
        y=np.asarray(data["y"], dtype=np.int64),
        patient_id=np.asarray(data["patient_id"]).astype(str),
        split=np.asarray(data["split"]).astype(str),
        window_start=np.asarray(data["window_start_date"]).astype(str),
        window_end=np.asarray(data["window_end_date"]).astype(str),
        continuous_feature_names=list(data.get("continuous_feature_names", [])),
        discrete_feature_names=list(data.get("discrete_feature_names", [])),
        integrated_feature_names=list(data.get("integrated_feature_names", [])),
        meta=dict(data.get("meta", {})),
    )
    # 거대한 flat 복제 배열은 DatasetBundle에 보관하지 않는다.
    del data
    gc.collect()
    return bundle


def _class_counts(values: Iterable[int]) -> dict[int, int]:
    return {int(key): int(value) for key, value in sorted(Counter(map(int, values)).items())}


def validate_data_contract(bundle: DatasetBundle) -> dict[str, Any]:
    arrays = {
        "X_continuous_seq": bundle.X_cont,
        "X_discrete_seq": bundle.X_disc,
        "X_integrated_seq": bundle.X_integrated,
        "y": bundle.y,
        "patient_id": bundle.patient_id,
        "split": bundle.split,
        "window_start_date": bundle.window_start,
        "window_end_date": bundle.window_end,
    }
    lengths = {name: len(value) for name, value in arrays.items()}
    if len(set(lengths.values())) != 1:
        raise ValueError(f"모든 배열 길이가 같아야 합니다: {lengths}")
    if bundle.X_cont.ndim != 4 or bundle.X_cont.shape[1:] != (7, 288, 5):
        raise ValueError(
            "X_continuous_seq shape 계약은 (N, 7, 288, 5)입니다: "
            f"{bundle.X_cont.shape}"
        )
    if bundle.X_disc.ndim != 3 or bundle.X_disc.shape[1] != 7:
        raise ValueError(
            "X_discrete_seq shape 계약은 (N, 7, n_discrete)입니다: "
            f"{bundle.X_disc.shape}"
        )
    if bundle.X_integrated.ndim != 3 or bundle.X_integrated.shape[:2] != (
        len(bundle.y),
        7,
    ):
        raise ValueError(
            "X_integrated_seq 첫 두 축 계약은 (N, 7)입니다: "
            f"{bundle.X_integrated.shape}"
        )
    if len(bundle.integrated_feature_names) != bundle.X_integrated.shape[-1]:
        raise ValueError(
            "integrated_feature_names 수와 X_integrated_seq feature 수가 다릅니다: "
            f"{len(bundle.integrated_feature_names)} vs {bundle.X_integrated.shape[-1]}"
        )
    if len(bundle.discrete_feature_names) != bundle.X_disc.shape[-1]:
        raise ValueError("discrete_feature_names 수와 X_discrete_seq feature 수가 다릅니다.")

    unique_y = set(map(int, np.unique(bundle.y)))
    if unique_y != {0, 1}:
        raise ValueError(f"라벨은 반드시 0=CN, 1=MCI+Dementia여야 합니다: {sorted(unique_y)}")
    unique_split = set(np.unique(bundle.split))
    if unique_split != {"train", "val"}:
        raise ValueError(f"split은 train/val이어야 합니다: {sorted(unique_split)}")
    missing_metadata = {
        name: int(np.isin(np.char.lower(values.astype(str)), ["", "nan", "none"]).sum())
        for name, values in {
            "patient_id": bundle.patient_id,
            "split": bundle.split,
            "window_start_date": bundle.window_start,
            "window_end_date": bundle.window_end,
        }.items()
    }
    if any(missing_metadata.values()):
        raise ValueError(f"window metadata에 결측 문자열이 있습니다: {missing_metadata}")

    numeric_arrays = {
        "X_continuous_seq": bundle.X_cont,
        "X_discrete_seq": bundle.X_disc,
        "X_integrated_seq": bundle.X_integrated,
    }
    non_finite = {name: int((~np.isfinite(value)).sum()) for name, value in numeric_arrays.items()}
    if any(non_finite.values()):
        raise ValueError(f"입력에 NaN/Inf가 있습니다: {non_finite}")

    frame = pd.DataFrame(
        {
            "patient_id": bundle.patient_id,
            "label": bundle.y,
            "split": bundle.split,
            "window_start_date": bundle.window_start,
            "window_end_date": bundle.window_end,
        }
    )
    conflicts = frame.groupby("patient_id")["label"].nunique()
    if (conflicts > 1).any():
        raise ValueError(f"환자별 라벨 충돌: {conflicts[conflicts > 1].index.tolist()[:10]}")
    train_subjects = set(frame.loc[frame["split"] == "train", "patient_id"])
    val_subjects = set(frame.loc[frame["split"] == "val", "patient_id"])
    overlap = sorted(train_subjects & val_subjects)
    if overlap:
        raise ValueError(f"train/공식 val patient_id 중복: {overlap[:10]}")
    duplicate_count = int(
        frame.duplicated(["patient_id", "window_start_date", "window_end_date"]).sum()
    )
    if duplicate_count:
        raise ValueError(f"동일 window key 중복: {duplicate_count}")

    metadata_padding = float(bundle.meta.get("padding_value", PADDING_VALUE))
    if metadata_padding != PADDING_VALUE:
        raise ValueError(
            f"continuous padding metadata가 -1.0이 아닙니다: {metadata_padding}"
        )
    padding_ratios = []
    for feature_index in range(bundle.X_cont.shape[-1]):
        feature_name = (
            bundle.continuous_feature_names[feature_index]
            if feature_index < len(bundle.continuous_feature_names)
            else f"continuous_{feature_index}"
        )
        ratio = float(np.mean(bundle.X_cont[..., feature_index] == PADDING_VALUE))
        padding_ratios.append({"feature": feature_name, "padding_ratio": ratio})
    if not np.any(bundle.X_cont == PADDING_VALUE):
        raise ValueError("continuous 입력에서 padding sentinel -1.0을 찾지 못했습니다.")
    valid_continuous = bundle.X_cont[bundle.X_cont != PADDING_VALUE]
    if valid_continuous.size and (
        valid_continuous.min() < -1e-6 or valid_continuous.max() > 1.0 + 1e-6
    ):
        raise ValueError("저장된 continuous non-padding 값이 train MinMax 범위 0~1을 벗어납니다.")

    split_summary: dict[str, Any] = {}
    print("\n=== 데이터 계약 검증 ===")
    print("shape:", {name: list(value.shape) for name, value in numeric_arrays.items()})
    print("NaN/Inf:", non_finite)
    for split_name in ("train", "val"):
        part = frame[frame["split"] == split_name]
        subjects = part.drop_duplicates("patient_id")
        split_summary[split_name] = {
            "windows": int(len(part)),
            "subjects": int(len(subjects)),
            "window_label_counts": _class_counts(part["label"]),
            "subject_label_counts": _class_counts(subjects["label"]),
        }
        print(split_name, split_summary[split_name])
    print("continuous 변수별 padding 비율:")
    print(pd.DataFrame(padding_ratios).to_string(index=False))
    print("동일 window key 중복:", duplicate_count)
    print("train/공식 val patient 중복:", len(overlap))

    return {
        "keys": list(arrays),
        "input_shapes": {name: list(value.shape) for name, value in numeric_arrays.items()},
        "array_lengths": lengths,
        "non_finite_counts": non_finite,
        "metadata_missing_counts": missing_metadata,
        "split_summary": split_summary,
        "padding_value": PADDING_VALUE,
        "continuous_padding_ratios": padding_ratios,
        "duplicate_window_keys": duplicate_count,
        "train_val_subject_overlap": len(overlap),
        "subject_label_conflicts": int((conflicts > 1).sum()),
    }


def choose_quick_train_subjects(
    y: np.ndarray,
    groups: np.ndarray,
    subjects_per_class: int,
    seed: int,
) -> set[str]:
    subject_frame = (
        pd.DataFrame({"patient_id": groups, "label": y})
        .drop_duplicates("patient_id")
        .sort_values("patient_id")
    )
    rng = np.random.default_rng(seed)
    selected: set[str] = set()
    for class_id in (0, 1):
        candidates = subject_frame.loc[
            subject_frame["label"] == class_id, "patient_id"
        ].to_numpy()
        rng.shuffle(candidates)
        selected.update(map(str, candidates[:subjects_per_class]))
    return selected


def split_experiment_arrays(
    bundle: DatasetBundle,
    config: ExperimentConfig,
) -> ExperimentArrays:
    train_mask = bundle.split == "train"
    if config.quick_mode:
        quick_subjects = choose_quick_train_subjects(
            bundle.y[train_mask],
            bundle.patient_id[train_mask],
            config.quick_subjects_per_class,
            config.seed,
        )
        train_mask &= np.isin(bundle.patient_id, list(quick_subjects))
        print(
            "QUICK_MODE: 공식 결과가 아닌 코드 경로 점검용입니다. "
            f"train subjects={len(quick_subjects)}"
        )
    val_mask = bundle.split == "val"
    arrays = ExperimentArrays(
        X_cont_train=bundle.X_cont[train_mask],
        X_disc_train=bundle.X_disc[train_mask],
        X_integrated_train=bundle.X_integrated[train_mask],
        y_train=bundle.y[train_mask],
        groups_train=bundle.patient_id[train_mask],
        start_train=bundle.window_start[train_mask],
        end_train=bundle.window_end[train_mask],
        X_cont_val=bundle.X_cont[val_mask],
        X_disc_val=bundle.X_disc[val_mask],
        X_integrated_val=bundle.X_integrated[val_mask],
        y_val=bundle.y[val_mask],
        groups_val=bundle.patient_id[val_mask],
        start_val=bundle.window_start[val_mask],
        end_val=bundle.window_end[val_mask],
        integrated_feature_names=bundle.integrated_feature_names,
    )
    return arrays


## Steps — 정규화, subject-safe fold, sample weight

기본값 `match_baseline`은 기존 최고 모델의 subject-history z-score 규칙을 적용합니다. 공식 train subject table에서만 4-fold를 만들고, subject class weight와 inverse-window weight를 결합합니다.


In [4]:
def baseline_feature_indices(feature_names: Sequence[str]) -> np.ndarray:
    missing = [name for name in BASELINE_SELECTED_FEATURES if name not in feature_names]
    if missing:
        raise KeyError(
            "기존 최고 LSTM의 선택 feature가 X_integrated_seq에 없습니다: "
            f"{missing}"
        )
    return np.asarray([feature_names.index(name) for name in BASELINE_SELECTED_FEATURES])


def per_subject_zscore(
    values: np.ndarray,
    groups: np.ndarray,
    *,
    padding_value: float | None,
    preserve_padding: bool,
) -> tuple[np.ndarray, int]:
    """기존 notebook의 subject-history z-score를 split 내부에서 재현한다.

    ``preserve_padding=True``는 새 continuous 모델용이다. 유효한 z-score가
    우연히 sentinel -1.0과 정확히 같아지면 mask 오인을 막기 위해 바로 옆의
    representable float로 이동시킨다.
    """
    values = np.asarray(values, dtype=np.float32)
    groups = np.asarray(groups).astype(str)
    output = (
        np.full_like(values, padding_value, dtype=np.float32)
        if preserve_padding and padding_value is not None
        else np.zeros_like(values, dtype=np.float32)
    )
    sentinel_collision_count = 0
    for subject_id in np.unique(groups):
        subject_indices = np.flatnonzero(groups == subject_id)
        block = values[subject_indices]
        output_block = output[subject_indices]
        for feature_index in range(values.shape[-1]):
            feature = block[..., feature_index]
            valid = np.isfinite(feature)
            if padding_value is not None:
                valid &= feature != padding_value
            valid_values = feature[valid]
            if valid_values.size < 2:
                if valid_values.size:
                    output_block[..., feature_index][valid] = 0.0
                continue
            mean = float(valid_values.mean())
            std = float(valid_values.std())
            normalized = np.zeros_like(feature, dtype=np.float32)
            if std >= 1e-8:
                normalized[valid] = (feature[valid] - mean) / std
            if preserve_padding and padding_value is not None:
                # mixed_float16에서도 유효값이 sentinel로 양자화되지 않도록 확인한다.
                collisions = valid & (
                    normalized.astype(np.float16) == np.float16(padding_value)
                )
                collision_count = int(collisions.sum())
                if collision_count:
                    normalized[collisions] = np.float32(
                        np.nextafter(
                            np.float16(padding_value),
                            np.float16(0.0),
                        )
                    )
                    sentinel_collision_count += collision_count
            output_block[..., feature_index][valid] = normalized[valid]
        output[subject_indices] = output_block
    return output, sentinel_collision_count


def prepare_training_inputs(
    arrays: ExperimentArrays,
    normalization_mode: str,
) -> tuple[dict[str, Any], dict[str, Any]]:
    """공식 train만 변환한다. 공식 val은 모델 선택이 끝난 뒤 별도로 변환한다."""
    selected_indices = baseline_feature_indices(arrays.integrated_feature_names)
    baseline = arrays.X_integrated_train[..., selected_indices]
    continuous = arrays.X_cont_train
    discrete = arrays.X_disc_train
    collisions = 0
    if normalization_mode == "match_baseline":
        baseline, _ = per_subject_zscore(
            baseline,
            arrays.groups_train,
            padding_value=PADDING_VALUE,
            preserve_padding=False,
        )
        continuous, collisions = per_subject_zscore(
            continuous,
            arrays.groups_train,
            padding_value=PADDING_VALUE,
            preserve_padding=True,
        )
        discrete, _ = per_subject_zscore(
            discrete,
            arrays.groups_train,
            padding_value=None,
            preserve_padding=False,
        )
    prepared = {
        MODEL_A: baseline,
        MODEL_B: [continuous, discrete],
        MODEL_C: [continuous, discrete],
    }
    details = {
        "mode": normalization_mode,
        "baseline_selected_features": BASELINE_SELECTED_FEATURES,
        "subject_history_zscore": normalization_mode == "match_baseline",
        "official_val_statistics_used_for_training": False,
        "continuous_padding_preserved": normalization_mode == "match_baseline",
        "valid_zscore_minus_one_collisions_nudged": collisions,
    }
    print("정규화:", json.dumps(details, ensure_ascii=False))
    return prepared, details


def prepare_official_val_input(
    model_name: str,
    arrays: ExperimentArrays,
    normalization_mode: str,
) -> Any:
    """선택 고정 후 공식 val 자체 subject history만 사용해 동일 규칙을 적용한다."""
    if model_name == MODEL_A:
        indices = baseline_feature_indices(arrays.integrated_feature_names)
        values = arrays.X_integrated_val[..., indices]
        if normalization_mode == "match_baseline":
            values, _ = per_subject_zscore(
                values,
                arrays.groups_val,
                padding_value=PADDING_VALUE,
                preserve_padding=False,
            )
        return values

    continuous = arrays.X_cont_val
    discrete = arrays.X_disc_val
    if normalization_mode == "match_baseline":
        continuous, _ = per_subject_zscore(
            continuous,
            arrays.groups_val,
            padding_value=PADDING_VALUE,
            preserve_padding=True,
        )
        discrete, _ = per_subject_zscore(
            discrete,
            arrays.groups_val,
            padding_value=None,
            preserve_padding=False,
        )
    return [continuous, discrete]


def subject_label_table(y: np.ndarray, groups: np.ndarray) -> pd.DataFrame:
    frame = pd.DataFrame(
        {"patient_id": np.asarray(groups).astype(str), "label": np.asarray(y, dtype=int)}
    )
    conflicts = frame.groupby("patient_id")["label"].nunique()
    if (conflicts > 1).any():
        raise ValueError("subject table 생성 중 환자별 라벨 충돌을 발견했습니다.")
    return (
        frame.groupby("patient_id", as_index=False)
        .agg(label=("label", "first"), window_count=("label", "size"))
        .sort_values("patient_id")
        .reset_index(drop=True)
    )


def build_subject_safe_folds(
    y: np.ndarray,
    groups: np.ndarray,
    config: ExperimentConfig,
) -> tuple[list[FoldDefinition], pd.DataFrame]:
    subjects = subject_label_table(y, groups)
    split_count = 2 if config.quick_mode else config.n_splits
    minimum_class_subjects = int(subjects["label"].value_counts().min())
    if minimum_class_subjects < split_count:
        raise ValueError(
            f"{split_count}-fold stratification에는 클래스별 subject가 최소 "
            f"{split_count}명 필요합니다: {subjects['label'].value_counts().to_dict()}"
        )
    splitter = StratifiedKFold(
        n_splits=split_count,
        shuffle=True,
        random_state=config.seed,
    )
    folds: list[FoldDefinition] = []
    manifest_rows: list[dict[str, Any]] = []
    for fold, (subject_train_pos, subject_val_pos) in enumerate(
        splitter.split(subjects["patient_id"], subjects["label"]), start=1
    ):
        train_subjects = subjects.iloc[subject_train_pos]["patient_id"].astype(str).tolist()
        val_subjects = subjects.iloc[subject_val_pos]["patient_id"].astype(str).tolist()
        if set(train_subjects) & set(val_subjects):
            raise AssertionError(f"fold {fold}: subject leakage")
        train_indices = np.flatnonzero(np.isin(groups, train_subjects))
        val_indices = np.flatnonzero(np.isin(groups, val_subjects))
        if set(groups[train_indices]) & set(groups[val_indices]):
            raise AssertionError(f"fold {fold}: window 확장 후 subject leakage")
        if set(np.unique(y[val_indices])) != {0, 1}:
            raise AssertionError(f"fold {fold}: validation fold에 두 클래스가 모두 없습니다.")
        folds.append(
            FoldDefinition(
                fold=fold,
                train_indices=train_indices,
                val_indices=val_indices,
                train_subjects=train_subjects,
                val_subjects=val_subjects,
            )
        )
        for role, positions in (("train", subject_train_pos), ("validation", subject_val_pos)):
            for row in subjects.iloc[positions].itertuples(index=False):
                manifest_rows.append(
                    {
                        "fold": fold,
                        "role": role,
                        "patient_id": row.patient_id,
                        "label": int(row.label),
                        "window_count": int(row.window_count),
                    }
                )
        validation_labels = subjects.iloc[subject_val_pos]["label"]
        print(
            f"fold={fold}: train subjects={len(train_subjects)}, "
            f"validation subjects={len(val_subjects)}, "
            f"validation labels={_class_counts(validation_labels)}"
        )
    return folds, pd.DataFrame(manifest_rows)


def subject_balanced_sample_weights(
    y: np.ndarray,
    groups: np.ndarray,
    *,
    inverse_window_count: bool,
    use_class_weight: bool,
) -> np.ndarray:
    """기존 LSTM의 subject class weight / subject window count 방식을 재현한다."""
    y = np.asarray(y, dtype=int)
    groups = np.asarray(groups).astype(str)
    subjects = subject_label_table(y, groups)
    class_counts = subjects["label"].value_counts().to_dict()
    class_weights = {0: 1.0, 1: 1.0}
    if use_class_weight:
        class_weights = {
            class_id: len(subjects) / (2.0 * int(class_counts[class_id]))
            for class_id in (0, 1)
        }
    window_counts = Counter(groups)
    weights = np.asarray(
        [
            class_weights[int(label)]
            / (window_counts[subject] if inverse_window_count else 1.0)
            for label, subject in zip(y, groups)
        ],
        dtype=np.float32,
    )
    return weights / weights.mean()


def slice_inputs(inputs: Any, indices: np.ndarray) -> Any:
    if isinstance(inputs, (list, tuple)):
        return [np.asarray(value)[indices] for value in inputs]
    return np.asarray(inputs)[indices]


def input_length(inputs: Any) -> int:
    return len(inputs[0]) if isinstance(inputs, (list, tuple)) else len(inputs)


## Steps — padding/mask와 세 모델 정의

Model A의 실제 체크포인트 구조를 재현하고, Model B/C는 continuous와 discrete 입력을 분리한 intermediate fusion으로 구현합니다. Conv는 날짜별로 먼저 실행되어 날짜 경계를 넘지 않습니다.


In [5]:
@keras.utils.register_keras_serializable(package="BinaryIntermediateFusion")
class ContinuousPaddingAndMask(layers.Layer):
    """-1.0 padding을 0으로 치환하고 변수별 observed mask 5개를 붙인다."""

    def __init__(self, padding_value: float = PADDING_VALUE, **kwargs: Any):
        super().__init__(**kwargs)
        self.padding_value = float(padding_value)

    def call(self, inputs: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
        observed = tf.not_equal(inputs, tf.cast(self.padding_value, inputs.dtype))
        clean = tf.where(observed, inputs, tf.zeros_like(inputs))
        model_inputs = tf.concat([clean, tf.cast(observed, inputs.dtype)], axis=-1)
        return model_inputs, observed

    def get_config(self) -> dict[str, Any]:
        return {**super().get_config(), "padding_value": self.padding_value}


@keras.utils.register_keras_serializable(package="BinaryIntermediateFusion")
class DownsampledTokenMask(layers.Layer):
    """12개 5분 구간 중 관측값이 하나라도 있으면 valid token으로 표시한다."""

    def __init__(self, pool_size: int = 12, **kwargs: Any):
        super().__init__(**kwargs)
        self.pool_size = int(pool_size)

    def call(self, observed_mask: tf.Tensor) -> tf.Tensor:
        time_observed = tf.reduce_any(observed_mask, axis=-1)
        shape = tf.shape(time_observed)
        daily = tf.reshape(
            tf.cast(time_observed, tf.float32),
            [shape[0] * shape[1], shape[2], 1],
        )
        pooled = tf.nn.max_pool1d(
            daily,
            ksize=self.pool_size,
            strides=self.pool_size,
            padding="SAME",
        )
        token_mask = tf.reshape(pooled > 0.0, [shape[0], -1])
        return token_mask

    def get_config(self) -> dict[str, Any]:
        return {**super().get_config(), "pool_size": self.pool_size}


@keras.utils.register_keras_serializable(package="BinaryIntermediateFusion")
class LearnablePositionEmbedding(layers.Layer):
    def build(self, input_shape: Sequence[int]) -> None:
        sequence_length = int(input_shape[1])
        embedding_dim = int(input_shape[2])
        self.position_embedding = self.add_weight(
            name="position_embedding",
            shape=(sequence_length, embedding_dim),
            initializer=keras.initializers.RandomNormal(stddev=0.02),
            trainable=True,
        )
        super().build(input_shape)

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        return inputs + tf.cast(self.position_embedding[None, ...], inputs.dtype)

    def get_config(self) -> dict[str, Any]:
        return super().get_config()


@keras.utils.register_keras_serializable(package="BinaryIntermediateFusion")
class TransformerEncoder(layers.Layer):
    """pre-LN residual Transformer encoder 1 block."""

    def __init__(
        self,
        d_model: int = 32,
        num_heads: int = 2,
        key_dim: int = 16,
        ff_dim: int = 64,
        dropout: float = 0.20,
        **kwargs: Any,
    ):
        super().__init__(**kwargs)
        self.d_model = int(d_model)
        self.num_heads = int(num_heads)
        self.key_dim = int(key_dim)
        self.ff_dim = int(ff_dim)
        self.dropout_rate = float(dropout)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attention = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.key_dim,
            dropout=self.dropout_rate,
        )
        self.attention_dropout = layers.Dropout(self.dropout_rate)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn_dense1 = layers.Dense(self.ff_dim, activation="swish")
        self.ffn_dropout1 = layers.Dropout(self.dropout_rate)
        self.ffn_dense2 = layers.Dense(self.d_model)
        self.ffn_dropout2 = layers.Dropout(self.dropout_rate)

    def call(self, inputs: Sequence[tf.Tensor], training: bool | None = None) -> tf.Tensor:
        tokens, token_mask = inputs
        token_mask = tf.cast(token_mask, tf.bool)
        batch_size = tf.shape(token_mask)[0]
        fallback = tf.concat(
            [
                tf.ones([batch_size, 1], dtype=tf.bool),
                tf.zeros([batch_size, tf.shape(token_mask)[1] - 1], dtype=tf.bool),
            ],
            axis=1,
        )
        any_valid = tf.reduce_any(token_mask, axis=1, keepdims=True)
        safe_key_mask = tf.where(any_valid, token_mask, fallback)
        attention_mask = safe_key_mask[:, tf.newaxis, :]

        normalized = self.norm1(tokens)
        attended = self.attention(
            normalized,
            normalized,
            attention_mask=attention_mask,
            training=training,
        )
        x = tokens + self.attention_dropout(attended, training=training)
        normalized = self.norm2(x)
        feed_forward = self.ffn_dense1(normalized)
        feed_forward = self.ffn_dropout1(feed_forward, training=training)
        feed_forward = self.ffn_dense2(feed_forward)
        feed_forward = self.ffn_dropout2(feed_forward, training=training)
        return x + feed_forward

    def get_config(self) -> dict[str, Any]:
        return {
            **super().get_config(),
            "d_model": self.d_model,
            "num_heads": self.num_heads,
            "key_dim": self.key_dim,
            "ff_dim": self.ff_dim,
            "dropout": self.dropout_rate,
        }


@keras.utils.register_keras_serializable(package="BinaryIntermediateFusion")
class MaskedGlobalAveragePooling1D(layers.Layer):
    """완전 결측 token을 제외하며 분모가 0이면 유한한 0 embedding을 반환한다."""

    def call(self, inputs: Sequence[tf.Tensor]) -> tf.Tensor:
        tokens, token_mask = inputs
        tokens = tf.cast(tokens, tf.float32)
        weights = tf.cast(token_mask, tf.float32)[..., tf.newaxis]
        numerator = tf.reduce_sum(tokens * weights, axis=1)
        denominator = tf.maximum(tf.reduce_sum(weights, axis=1), 1.0)
        return numerator / denominator

    def get_config(self) -> dict[str, Any]:
        return super().get_config()


def build_baseline_model(config: ExperimentConfig) -> keras.Model:
    """실제 final_model.keras 구조(4,849 params)를 그대로 재구현한다."""
    model_input = keras.Input(shape=(7, len(BASELINE_SELECTED_FEATURES)), name="integrated_top20")
    regularizer = regularizers.l2(5e-3)
    x = layers.Conv1D(
        32,
        3,
        padding="same",
        kernel_regularizer=regularizer,
        name="baseline_conv1d",
    )(model_input)
    x = layers.BatchNormalization(name="baseline_batch_norm")(x)
    x = layers.Activation("relu", name="baseline_relu")(x)
    x = layers.Bidirectional(
        layers.LSTM(
            8,
            return_sequences=True,
            kernel_regularizer=regularizer,
            recurrent_regularizer=regularizer,
        ),
        name="baseline_bidirectional_lstm",
    )(x)
    x = layers.GlobalAveragePooling1D(name="baseline_global_average")(x)
    x = layers.Dense(
        8,
        activation="relu",
        kernel_regularizer=regularizer,
        name="baseline_dense",
    )(x)
    x = layers.Dropout(0.50, name="baseline_dropout")(x)
    output = layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        kernel_regularizer=regularizer,
        name="binary_output",
    )(x)
    model = keras.Model(model_input, output, name=MODEL_A)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config.baseline_learning_rate),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.AUC(name="window_auc"),
        ],
    )
    if model.count_params() != 4_849:
        raise AssertionError(
            f"기존 final_model.keras와 parameter 수가 다릅니다: {model.count_params()} vs 4849"
        )
    return model


def _continuous_branch(
    continuous_input: keras.KerasTensor,
    *,
    use_transformer: bool,
) -> tuple[keras.KerasTensor, keras.KerasTensor]:
    # TimeDistributed Conv를 먼저 적용하므로 서로 다른 날짜의 경계를 넘지 않는다.
    model_inputs, observed_mask = ContinuousPaddingAndMask(
        name="continuous_padding_and_missingness"
    )(continuous_input)
    x = layers.TimeDistributed(
        layers.SeparableConv1D(
            filters=16,
            kernel_size=5,
            padding="same",
            activation="swish",
        ),
        name="daywise_separable_conv",
    )(model_inputs)
    x = layers.TimeDistributed(
        layers.Conv1D(
            filters=32,
            kernel_size=12,
            strides=12,
            padding="same",
            activation="swish",
        ),
        name="daywise_downsample_conv",
    )(x)
    tokens = layers.Reshape((7 * 24, 32), name="flatten_day_tokens")(x)
    token_mask = DownsampledTokenMask(name="token_validity_mask")(observed_mask)
    if use_transformer:
        tokens = LearnablePositionEmbedding(name="learnable_position_embedding")(tokens)
        tokens = TransformerEncoder(
            d_model=32,
            num_heads=2,
            key_dim=16,
            ff_dim=64,
            dropout=0.20,
            name="transformer_encoder_block_1",
        )([tokens, token_mask])
    pooled = MaskedGlobalAveragePooling1D(name="masked_global_average")(
        [tokens, token_mask]
    )
    embedding = layers.Dense(32, activation="swish", name="continuous_embedding_dense")(
        pooled
    )
    embedding = layers.Dropout(0.30, name="continuous_embedding_dropout")(embedding)
    return embedding, token_mask


def build_intermediate_model(
    n_discrete: int,
    config: ExperimentConfig,
    *,
    use_transformer: bool,
) -> keras.Model:
    continuous_input = keras.Input(shape=(7, 288, 5), name="continuous_input")
    discrete_input = keras.Input(shape=(7, n_discrete), name="discrete_input")
    continuous_embedding, _ = _continuous_branch(
        continuous_input,
        use_transformer=use_transformer,
    )

    discrete = layers.LayerNormalization(name="discrete_layer_norm")(discrete_input)
    discrete = layers.TimeDistributed(
        layers.Dense(16, activation="swish"),
        name="discrete_day_dense",
    )(discrete)
    discrete = layers.GlobalAveragePooling1D(name="discrete_global_average")(discrete)
    discrete = layers.Dense(8, activation="swish", name="discrete_embedding_dense")(
        discrete
    )
    discrete_embedding = layers.Dropout(0.20, name="discrete_embedding_dropout")(
        discrete
    )

    fused = layers.Concatenate(name="intermediate_fusion")(
        [continuous_embedding, discrete_embedding]
    )
    fused = layers.Dense(16, activation="swish", name="fusion_dense")(fused)
    fused = layers.Dropout(0.35, name="fusion_dropout")(fused)
    output = layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="binary_output",
    )(fused)
    model_name = MODEL_C if use_transformer else MODEL_B
    model = keras.Model(
        inputs=[continuous_input, discrete_input],
        outputs=output,
        name=model_name,
    )
    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=config.learning_rate,
            clipnorm=config.clipnorm,
        ),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.AUC(name="window_auc")],
    )
    expected_parameters = 8_283 + 18 * int(n_discrete)
    if use_transformer:
        expected_parameters += 13_920
    if model.count_params() != expected_parameters:
        raise AssertionError(
            f"{model_name} architecture parameter 수가 예상과 다릅니다: "
            f"{model.count_params()} vs {expected_parameters}"
        )
    return model


def build_model(model_name: str, n_discrete: int, config: ExperimentConfig) -> keras.Model:
    if model_name == MODEL_A:
        return build_baseline_model(config)
    if model_name == MODEL_B:
        return build_intermediate_model(n_discrete, config, use_transformer=False)
    if model_name == MODEL_C:
        return build_intermediate_model(n_discrete, config, use_transformer=True)
    raise ValueError(f"알 수 없는 model_name: {model_name}")


def trainable_parameter_count(model: keras.Model) -> int:
    return int(sum(np.prod(variable.shape) for variable in model.trainable_weights))


def run_mask_unit_tests() -> dict[str, Any]:
    """mask shape, 12-step 집계, all-missing pooling/attention 안정성을 확인한다."""
    sample = np.full((2, 7, 288, 5), PADDING_VALUE, dtype=np.float32)
    sample[0, 0, 0, 0] = 0.25
    sample[0, 6, 287, 4] = 0.75
    model_inputs, observed = ContinuousPaddingAndMask()(tf.constant(sample))
    token_mask = DownsampledTokenMask()(observed)
    if tuple(model_inputs.shape) != (2, 7, 288, 10):
        raise AssertionError(f"missingness 결합 shape 오류: {model_inputs.shape}")
    if tuple(token_mask.shape) != (2, 168):
        raise AssertionError(f"token mask shape 오류: {token_mask.shape}")
    mask_values = token_mask.numpy()
    if int(mask_values[0].sum()) != 2 or int(mask_values[1].sum()) != 0:
        raise AssertionError(f"12-step token mask 값 오류: {mask_values.sum(axis=1)}")
    dummy_tokens = tf.ones((2, 168, 32), dtype=tf.float32)
    encoded = TransformerEncoder()([dummy_tokens, token_mask], training=False)
    pooled = MaskedGlobalAveragePooling1D()([encoded, token_mask])
    if tuple(encoded.shape) != (2, 168, 32) or tuple(pooled.shape) != (2, 32):
        raise AssertionError("Transformer/masked pooling shape 오류")
    if not np.isfinite(pooled.numpy()).all():
        raise AssertionError("all-missing token에서 NaN/Inf가 발생했습니다.")
    result = {
        "x_model_shape": list(model_inputs.shape),
        "token_mask_shape": list(token_mask.shape),
        "valid_tokens_by_sample": mask_values.sum(axis=1).astype(int).tolist(),
        "all_missing_pooling_is_finite": True,
        "date_boundary_convolution": "TimeDistributed before reshape",
    }
    print("mask unit test 통과:", result)
    return result


def inspect_model_architectures(
    n_discrete: int,
    config: ExperimentConfig,
    result_dir: Path,
) -> dict[str, dict[str, Any]]:
    summaries: list[str] = []
    architectures: dict[str, dict[str, Any]] = {}
    for model_name in MODEL_ORDER:
        keras.backend.clear_session()
        model = build_model(model_name, n_discrete, config)
        summary_lines: list[str] = []
        model.summary(print_fn=summary_lines.append)
        summaries.extend([f"\n{'=' * 88}\n{model_name}\n{'=' * 88}", *summary_lines])
        architectures[model_name] = {
            "input_shape": json_ready(model.input_shape),
            "output_shape": json_ready(model.output_shape),
            "total_parameters": int(model.count_params()),
            "trainable_parameters": trainable_parameter_count(model),
        }
        print(
            f"{model_name}: total_parameters={architectures[model_name]['total_parameters']:,}, "
            f"trainable_parameters={architectures[model_name]['trainable_parameters']:,}"
        )
        del model
        keras.backend.clear_session()
        gc.collect()
    (result_dir / "model_summary.txt").write_text("\n".join(summaries), encoding="utf-8")
    return architectures


## Steps — subject-level 지표와 early stopping

기존 기준선에 명시된 window probability 평균으로 subject probability를 집계합니다. subject ROC-AUC를 early stopping 주 기준으로, balanced accuracy를 동률 기준으로 사용합니다.


In [6]:
def aggregate_subject_probabilities(
    y: np.ndarray,
    probabilities: np.ndarray,
    groups: np.ndarray,
) -> pd.DataFrame:
    """기존 최고 LSTM과 같은 window probability 산술평균을 사용한다."""
    frame = pd.DataFrame(
        {
            "patient_id": np.asarray(groups).astype(str),
            "label": np.asarray(y, dtype=int),
            "probability": np.asarray(probabilities, dtype=float).reshape(-1),
        }
    )
    conflicts = frame.groupby("patient_id")["label"].nunique()
    if (conflicts > 1).any():
        raise ValueError("subject probability 집계 중 라벨 충돌을 발견했습니다.")
    return (
        frame.groupby("patient_id", as_index=False)
        .agg(
            label=("label", "first"),
            probability=("probability", "mean"),
            n_windows=("probability", "size"),
        )
        .sort_values("patient_id")
        .reset_index(drop=True)
    )


def safe_roc_auc(y_true: np.ndarray, probabilities: np.ndarray) -> float:
    return (
        float(roc_auc_score(y_true, probabilities))
        if len(np.unique(y_true)) == 2
        else float("nan")
    )


def binary_metrics(
    y_true: Sequence[int],
    probabilities: Sequence[float],
    threshold: float,
) -> dict[str, Any]:
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predicted = (probabilities >= float(threshold)).astype(int)
    matrix = confusion_matrix(y_true, predicted, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    specificity = float(tn / max(tn + fp, 1))
    return {
        "roc_auc": safe_roc_auc(y_true, probabilities),
        "pr_auc": float(average_precision_score(y_true, probabilities)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predicted)),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "sensitivity": float(recall_score(y_true, predicted, zero_division=0)),
        "specificity": specificity,
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, predicted)),
        "brier_score": float(brier_score_loss(y_true, probabilities)),
        "threshold": float(threshold),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "confusion_matrix": matrix.tolist(),
    }


def choose_oof_threshold(
    y_true: Sequence[int],
    probabilities: Sequence[float],
) -> tuple[float, pd.DataFrame]:
    """train OOF에서 balanced accuracy, Youden J, 0.5 근접 순으로 선택한다."""
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    candidates = np.unique(
        np.concatenate(
            [
                np.linspace(0.01, 0.99, 197),
                probabilities,
                np.asarray([0.5]),
            ]
        )
    )
    rows = []
    for threshold in candidates:
        predicted = (probabilities >= threshold).astype(int)
        matrix = confusion_matrix(y_true, predicted, labels=[0, 1])
        tn, fp, fn, tp = matrix.ravel()
        sensitivity = tp / max(tp + fn, 1)
        specificity = tn / max(tn + fp, 1)
        rows.append(
            {
                "threshold": float(threshold),
                "balanced_accuracy": float((sensitivity + specificity) / 2),
                "youden_j": float(sensitivity + specificity - 1),
                "distance_from_0_5": float(abs(threshold - 0.5)),
            }
        )
    table = pd.DataFrame(rows).sort_values(
        ["balanced_accuracy", "youden_j", "distance_from_0_5", "threshold"],
        ascending=[False, False, True, True],
    )
    return float(table.iloc[0]["threshold"]), table.reset_index(drop=True)


class SubjectAUCMonitor(keras.callbacks.Callback):
    """subject ROC-AUC를 주 기준, @0.5 balanced accuracy를 동률 기준으로 본다."""

    def __init__(
        self,
        validation_inputs: Any,
        validation_y: np.ndarray,
        validation_groups: np.ndarray,
        *,
        patience: int,
        batch_size: int = 256,
    ):
        super().__init__()
        self.validation_inputs = validation_inputs
        self.validation_y = np.asarray(validation_y, dtype=int)
        self.validation_groups = np.asarray(validation_groups).astype(str)
        self.patience = int(patience)
        self.batch_size = int(batch_size)
        self.best_key = (-np.inf, -np.inf)
        self.best_epoch = 0
        self.best_weights: list[np.ndarray] | None = None
        self.wait = 0

    def on_epoch_end(self, epoch: int, logs: dict[str, Any] | None = None) -> None:
        probabilities = self.model.predict(
            self.validation_inputs,
            batch_size=self.batch_size,
            verbose=0,
        ).reshape(-1)
        subjects = aggregate_subject_probabilities(
            self.validation_y,
            probabilities,
            self.validation_groups,
        )
        metrics = binary_metrics(subjects["label"], subjects["probability"], 0.5)
        auc = metrics["roc_auc"]
        balanced_accuracy = metrics["balanced_accuracy"]
        if logs is not None:
            logs["val_subject_roc_auc"] = auc
            logs["val_subject_balanced_accuracy"] = balanced_accuracy
        key = (auc, balanced_accuracy)
        if key > self.best_key:
            self.best_key = key
            self.best_epoch = epoch + 1
            self.best_weights = self.model.get_weights()
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True

    def on_train_end(self, logs: dict[str, Any] | None = None) -> None:
        if self.best_weights is None:
            raise RuntimeError("subject ROC-AUC best weights를 저장하지 못했습니다.")
        self.model.set_weights(self.best_weights)


def model_max_epochs(model_name: str, config: ExperimentConfig) -> int:
    if config.quick_mode:
        return 3
    return config.baseline_max_epochs if model_name == MODEL_A else config.max_epochs


def make_training_callbacks(
    validation_inputs: Any,
    validation_y: np.ndarray,
    validation_groups: np.ndarray,
    config: ExperimentConfig,
) -> tuple[SubjectAUCMonitor, list[keras.callbacks.Callback]]:
    patience = 1 if config.quick_mode else config.patience
    monitor = SubjectAUCMonitor(
        validation_inputs,
        validation_y,
        validation_groups,
        patience=patience,
    )
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=max(1, min(4, patience // 2)),
        min_lr=1e-5,
        verbose=0,
    )
    return monitor, [monitor, reduce_lr, keras.callbacks.TerminateOnNaN()]


def save_and_reload_fold_model(
    model: keras.Model,
    model_path: Path,
    validation_inputs: Any,
) -> tuple[int, bool]:
    model.save(model_path)
    probe_count = min(8, input_length(validation_inputs))
    probe_indices = np.arange(probe_count)
    probe_inputs = slice_inputs(validation_inputs, probe_indices)
    before = model.predict(probe_inputs, verbose=0).reshape(-1)
    reloaded = keras.models.load_model(model_path, compile=False)
    after = reloaded.predict(probe_inputs, verbose=0).reshape(-1)
    verified = bool(np.allclose(before, after, rtol=2e-4, atol=2e-5))
    if not verified:
        raise AssertionError(f"저장/재로드 prediction 불일치: {model_path}")
    del reloaded
    return int(model_path.stat().st_size), verified


def train_fold(
    model_name: str,
    fold: FoldDefinition,
    prepared_inputs: Any,
    y_train: np.ndarray,
    groups_train: np.ndarray,
    n_discrete: int,
    config: ExperimentConfig,
    result_dir: Path,
    model_index: int,
) -> dict[str, Any]:
    keras.backend.clear_session()
    gc.collect()
    fold_seed = config.seed + model_index * 1_000 + fold.fold
    set_global_seed(fold_seed)
    model = build_model(model_name, n_discrete, config)
    train_inputs = slice_inputs(prepared_inputs, fold.train_indices)
    validation_inputs = slice_inputs(prepared_inputs, fold.val_indices)
    sample_weight = subject_balanced_sample_weights(
        y_train[fold.train_indices],
        groups_train[fold.train_indices],
        inverse_window_count=config.inverse_subject_window_weight,
        use_class_weight=config.subject_class_weight,
    )
    monitor, callbacks = make_training_callbacks(
        validation_inputs,
        y_train[fold.val_indices],
        groups_train[fold.val_indices],
        config,
    )
    started = time.perf_counter()
    history = model.fit(
        train_inputs,
        y_train[fold.train_indices],
        sample_weight=sample_weight,
        validation_data=(validation_inputs, y_train[fold.val_indices]),
        epochs=model_max_epochs(model_name, config),
        batch_size=config.batch_size,
        callbacks=callbacks,
        shuffle=True,
        verbose=2 if config.quick_mode else 0,
    )
    training_seconds = float(time.perf_counter() - started)
    probabilities = model.predict(validation_inputs, batch_size=256, verbose=0).reshape(-1)
    subject_frame = aggregate_subject_probabilities(
        y_train[fold.val_indices],
        probabilities,
        groups_train[fold.val_indices],
    )
    metrics = binary_metrics(subject_frame["label"], subject_frame["probability"], 0.5)

    history_path = result_dir / (
        f"training_history_fold_{fold.fold}_{MODEL_SHORT_NAME[model_name]}.csv"
    )
    pd.DataFrame(history.history).to_csv(history_path, index=False)
    fold_model_dir = result_dir / "fold_models"
    fold_model_dir.mkdir(exist_ok=True)
    model_path = fold_model_dir / f"{MODEL_SHORT_NAME[model_name]}_fold_{fold.fold}.keras"
    model_size_bytes, reload_verified = save_and_reload_fold_model(
        model,
        model_path,
        validation_inputs,
    )
    result = {
        "probabilities": probabilities.astype(np.float32),
        "metrics_05": metrics,
        "best_epoch": int(monitor.best_epoch),
        "training_seconds": training_seconds,
        "total_parameters": int(model.count_params()),
        "trainable_parameters": trainable_parameter_count(model),
        "model_size_bytes": model_size_bytes,
        "reload_prediction_verified": reload_verified,
        "fold_seed": fold_seed,
    }
    print(
        f"{model_name} fold={fold.fold}: subject_auc={metrics['roc_auc']:.4f}, "
        f"subject_pr_auc={metrics['pr_auc']:.4f}, best_epoch={monitor.best_epoch}, "
        f"seconds={training_seconds:.1f}"
    )
    del model, history, train_inputs, validation_inputs, sample_weight
    keras.backend.clear_session()
    gc.collect()
    return result


## Steps — 동일 fold OOF 비교와 모델 선택

세 모델을 같은 fold로 학습해 OOF prediction을 저장합니다. ROC-AUC, PR-AUC, balanced accuracy, fold 표준편차 순으로 비교하며 Transformer 우위가 불명확하면 자동 선택하지 않습니다.


In [7]:
def select_final_model(
    comparison: pd.DataFrame,
    transformer_min_auc_gain: float,
) -> tuple[str, dict[str, Any]]:
    ranked = comparison.copy()
    ranked = ranked.sort_values(
        [
            "subject_roc_auc",
            "subject_pr_auc",
            "subject_balanced_accuracy_at_05",
            "fold_subject_roc_auc_std",
        ],
        ascending=[False, False, False, True],
    )
    raw_winner = str(ranked.iloc[0]["model"])
    selected = raw_winner
    transformer_guard_applied = False
    rationale = "사전 정의한 ROC-AUC > PR-AUC > balanced accuracy > fold std 기준"
    if raw_winner == MODEL_C:
        non_transformer = ranked[ranked["model"] != MODEL_C].copy()
        best_non_transformer = non_transformer.iloc[0]
        transformer = ranked.iloc[0]
        auc_gain = float(
            transformer["subject_roc_auc"] - best_non_transformer["subject_roc_auc"]
        )
        pr_not_worse = bool(
            transformer["subject_pr_auc"] >= best_non_transformer["subject_pr_auc"]
        )
        if auc_gain < transformer_min_auc_gain or not pr_not_worse:
            selected = str(best_non_transformer["model"])
            transformer_guard_applied = True
            rationale = (
                "Transformer 우위가 명확하지 않아 보수적으로 최고 non-Transformer 선택: "
                f"AUC gain={auc_gain:.4f} (필요 {transformer_min_auc_gain:.4f}), "
                f"PR-AUC non-inferior={pr_not_worse}"
            )
    return selected, {
        "raw_winner": raw_winner,
        "selected_model": selected,
        "transformer_guard_applied": transformer_guard_applied,
        "transformer_min_auc_gain": transformer_min_auc_gain,
        "rationale": rationale,
    }


def run_oof_comparison(
    arrays: ExperimentArrays,
    prepared_by_model: dict[str, Any],
    folds: list[FoldDefinition],
    architectures: dict[str, dict[str, Any]],
    config: ExperimentConfig,
    result_dir: Path,
) -> dict[str, Any]:
    fold_by_window = np.full(len(arrays.y_train), -1, dtype=int)
    for fold in folds:
        fold_by_window[fold.val_indices] = fold.fold
    if np.any(fold_by_window < 1):
        raise AssertionError("모든 train window가 정확히 한 validation fold에 있어야 합니다.")

    fold_metric_rows: list[dict[str, Any]] = []
    oof_window_frames: list[pd.DataFrame] = []
    oof_subject_frames: list[pd.DataFrame] = []
    comparison_rows: list[dict[str, Any]] = []
    window_metric_rows: list[dict[str, Any]] = []
    model_results: dict[str, Any] = {}

    for model_index, model_name in enumerate(MODEL_ORDER, start=1):
        print(f"\n=== OOF: {model_name} ===")
        oof_probabilities = np.full(len(arrays.y_train), np.nan, dtype=np.float32)
        fold_results = []
        for fold in folds:
            result = train_fold(
                model_name,
                fold,
                prepared_by_model[model_name],
                arrays.y_train,
                arrays.groups_train,
                arrays.X_disc_train.shape[-1],
                config,
                result_dir,
                model_index,
            )
            oof_probabilities[fold.val_indices] = result["probabilities"]
            fold_results.append(result)
        if not np.isfinite(oof_probabilities).all():
            raise AssertionError(f"OOF prediction 누락/비정상: {model_name}")

        subject_frame = aggregate_subject_probabilities(
            arrays.y_train,
            oof_probabilities,
            arrays.groups_train,
        )
        subject_to_fold = pd.DataFrame(
            {"patient_id": arrays.groups_train, "fold": fold_by_window}
        ).drop_duplicates("patient_id")
        if (subject_to_fold.groupby("patient_id")["fold"].nunique() > 1).any():
            raise AssertionError("한 subject의 OOF window가 여러 fold에 있습니다.")
        subject_frame = subject_frame.merge(subject_to_fold, on="patient_id", how="left")
        threshold, threshold_grid = choose_oof_threshold(
            subject_frame["label"], subject_frame["probability"]
        )
        threshold_grid.to_csv(
            result_dir / f"threshold_grid_{MODEL_SHORT_NAME[model_name]}.csv",
            index=False,
        )
        subject_metrics_05 = binary_metrics(
            subject_frame["label"], subject_frame["probability"], 0.5
        )
        subject_metrics_tuned = binary_metrics(
            subject_frame["label"], subject_frame["probability"], threshold
        )
        window_metrics_05 = binary_metrics(arrays.y_train, oof_probabilities, 0.5)
        window_metrics_tuned = binary_metrics(arrays.y_train, oof_probabilities, threshold)
        for threshold_type, metrics in (
            ("fixed_0.5", window_metrics_05),
            ("model_oof_selected", window_metrics_tuned),
        ):
            window_metric_rows.append(
                {
                    "model": model_name,
                    "scope": "train_oof_window_reference_only",
                    "threshold_type": threshold_type,
                    **{key: value for key, value in metrics.items() if key != "confusion_matrix"},
                    "confusion_matrix": json.dumps(metrics["confusion_matrix"]),
                }
            )

        for fold, result in zip(folds, fold_results):
            fold_subjects = subject_frame[subject_frame["fold"] == fold.fold]
            for threshold_name, threshold_value in (
                ("fixed_0.5", 0.5),
                ("model_oof_selected", threshold),
            ):
                metrics = binary_metrics(
                    fold_subjects["label"],
                    fold_subjects["probability"],
                    threshold_value,
                )
                fold_metric_rows.append(
                    {
                        "model": model_name,
                        "fold": fold.fold,
                        "threshold_type": threshold_name,
                        "best_epoch": result["best_epoch"],
                        "training_seconds": result["training_seconds"],
                        "total_parameters": result["total_parameters"],
                        "trainable_parameters": result["trainable_parameters"],
                        "model_size_bytes": result["model_size_bytes"],
                        "reload_prediction_verified": result[
                            "reload_prediction_verified"
                        ],
                        **{key: value for key, value in metrics.items() if key != "confusion_matrix"},
                        "confusion_matrix": json.dumps(metrics["confusion_matrix"]),
                    }
                )

        window_frame = pd.DataFrame(
            {
                "model": model_name,
                "window_index": np.arange(len(arrays.y_train)),
                "fold": fold_by_window,
                "patient_id": arrays.groups_train,
                "window_start_date": arrays.start_train,
                "window_end_date": arrays.end_train,
                "label": arrays.y_train,
                "probability": oof_probabilities,
                "pred_fixed_0_5": (oof_probabilities >= 0.5).astype(int),
                "pred_oof_threshold": (oof_probabilities >= threshold).astype(int),
            }
        )
        subject_frame.insert(0, "model", model_name)
        subject_frame["pred_fixed_0_5"] = (subject_frame["probability"] >= 0.5).astype(int)
        subject_frame["pred_oof_threshold"] = (
            subject_frame["probability"] >= threshold
        ).astype(int)
        subject_frame["oof_threshold"] = threshold
        oof_window_frames.append(window_frame)
        oof_subject_frames.append(subject_frame)

        fixed_fold_rows = [
            row
            for row in fold_metric_rows
            if row["model"] == model_name and row["threshold_type"] == "fixed_0.5"
        ]
        fold_summary: dict[str, float] = {}
        for metric_name in (
            "roc_auc",
            "pr_auc",
            "balanced_accuracy",
            "f1",
            "sensitivity",
            "specificity",
            "precision",
            "accuracy",
            "brier_score",
        ):
            metric_values = np.asarray(
                [row[metric_name] for row in fixed_fold_rows], dtype=float
            )
            fold_summary[f"fold_subject_{metric_name}_mean"] = float(
                np.mean(metric_values)
            )
            fold_summary[f"fold_subject_{metric_name}_std"] = float(
                np.std(metric_values, ddof=1)
            )
        best_epochs = [result["best_epoch"] for result in fold_results]
        model_sizes = [result["model_size_bytes"] for result in fold_results]
        training_seconds = [result["training_seconds"] for result in fold_results]
        comparison_rows.append(
            {
                "model": model_name,
                "subject_roc_auc": subject_metrics_05["roc_auc"],
                "subject_pr_auc": subject_metrics_05["pr_auc"],
                "subject_balanced_accuracy_at_05": subject_metrics_05[
                    "balanced_accuracy"
                ],
                "subject_f1_at_05": subject_metrics_05["f1"],
                "subject_sensitivity_at_05": subject_metrics_05["sensitivity"],
                "subject_specificity_at_05": subject_metrics_05["specificity"],
                "subject_precision_at_05": subject_metrics_05["precision"],
                "subject_accuracy_at_05": subject_metrics_05["accuracy"],
                "subject_brier_score": subject_metrics_05["brier_score"],
                "oof_selected_threshold": threshold,
                "subject_balanced_accuracy_at_oof_threshold": subject_metrics_tuned[
                    "balanced_accuracy"
                ],
                "subject_f1_at_oof_threshold": subject_metrics_tuned["f1"],
                "subject_sensitivity_at_oof_threshold": subject_metrics_tuned[
                    "sensitivity"
                ],
                "subject_specificity_at_oof_threshold": subject_metrics_tuned[
                    "specificity"
                ],
                "subject_precision_at_oof_threshold": subject_metrics_tuned["precision"],
                "subject_accuracy_at_oof_threshold": subject_metrics_tuned["accuracy"],
                "window_roc_auc_reference": window_metrics_05["roc_auc"],
                "window_pr_auc_reference": window_metrics_05["pr_auc"],
                "window_balanced_accuracy_at_05_reference": window_metrics_05[
                    "balanced_accuracy"
                ],
                "window_balanced_accuracy_at_oof_threshold_reference": window_metrics_tuned[
                    "balanced_accuracy"
                ],
                **fold_summary,
                "median_best_epoch": int(round(float(np.median(best_epochs)))),
                "total_parameters": architectures[model_name]["total_parameters"],
                "trainable_parameters": architectures[model_name]["trainable_parameters"],
                "cv_training_seconds": float(np.sum(training_seconds)),
                "model_file_size_bytes_mean": int(round(float(np.mean(model_sizes)))),
                "model_file_size_mb_mean": float(np.mean(model_sizes) / 1024**2),
                "reload_prediction_verified_all_folds": bool(
                    all(result["reload_prediction_verified"] for result in fold_results)
                ),
            }
        )
        model_results[model_name] = {
            "oof_probabilities": oof_probabilities,
            "subject_frame": subject_frame,
            "threshold": threshold,
            "subject_metrics_05": subject_metrics_05,
            "subject_metrics_tuned": subject_metrics_tuned,
            "best_epochs": best_epochs,
        }

    fold_metrics = pd.DataFrame(fold_metric_rows)
    window_metrics = pd.DataFrame(window_metric_rows)
    comparison = pd.DataFrame(comparison_rows)
    comparison = comparison.sort_values(
        [
            "subject_roc_auc",
            "subject_pr_auc",
            "subject_balanced_accuracy_at_05",
            "fold_subject_roc_auc_std",
        ],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    oof_windows = pd.concat(oof_window_frames, ignore_index=True)
    oof_subjects = pd.concat(oof_subject_frames, ignore_index=True)
    fold_metrics.to_csv(result_dir / "fold_metrics.csv", index=False)
    window_metrics.to_csv(result_dir / "window_metrics_reference.csv", index=False)
    comparison.to_csv(result_dir / "model_comparison.csv", index=False)
    oof_windows.to_csv(result_dir / "oof_window_predictions.csv", index=False)
    oof_subjects.to_csv(result_dir / "oof_subject_predictions.csv", index=False)
    print("\n=== Model comparison (subject-level primary) ===")
    print(comparison.to_string(index=False))

    selected_model, selection_details = select_final_model(
        comparison,
        config.transformer_min_auc_gain,
    )
    print("선택:", json.dumps(selection_details, ensure_ascii=False))
    return {
        "model_results": model_results,
        "fold_metrics": fold_metrics,
        "window_metrics": window_metrics,
        "comparison": comparison,
        "oof_windows": oof_windows,
        "oof_subjects": oof_subjects,
        "selected_model": selected_model,
        "selection_details": selection_details,
    }


## Steps — 최종 refit, 공식 validation, 그림과 manifest

선택 모델의 fold best epoch 중앙값으로 공식 train 전체를 refit합니다. 저장 모델을 다시 로드한 뒤 고정 threshold로 공식 validation을 한 번 평가하고 모든 지정 산출물을 저장합니다.


In [8]:
def fit_selected_final_model(
    selected_model: str,
    final_epochs: int,
    prepared_train_inputs: Any,
    arrays: ExperimentArrays,
    config: ExperimentConfig,
    result_dir: Path,
) -> tuple[keras.Model, dict[str, Any]]:
    keras.backend.clear_session()
    gc.collect()
    set_global_seed(config.seed)
    model = build_model(selected_model, arrays.X_disc_train.shape[-1], config)
    weights = subject_balanced_sample_weights(
        arrays.y_train,
        arrays.groups_train,
        inverse_window_count=config.inverse_subject_window_weight,
        use_class_weight=config.subject_class_weight,
    )
    callbacks = [
        keras.callbacks.ReduceLROnPlateau(
            monitor="loss",
            mode="min",
            factor=0.5,
            patience=4,
            min_lr=1e-5,
            verbose=0,
        ),
        keras.callbacks.TerminateOnNaN(),
    ]
    started = time.perf_counter()
    history = model.fit(
        prepared_train_inputs,
        arrays.y_train,
        sample_weight=weights,
        epochs=final_epochs,
        batch_size=config.batch_size,
        callbacks=callbacks,
        shuffle=True,
        verbose=2 if config.quick_mode else 0,
    )
    training_seconds = float(time.perf_counter() - started)
    pd.DataFrame(history.history).to_csv(
        result_dir / "training_history_final_refit.csv", index=False
    )
    model_path = result_dir / "best_model.keras"
    probe_indices = np.arange(min(8, input_length(prepared_train_inputs)))
    probe_inputs = slice_inputs(prepared_train_inputs, probe_indices)
    probe_before = model.predict(probe_inputs, verbose=0).reshape(-1)
    model.save(model_path)
    del model
    keras.backend.clear_session()
    gc.collect()
    reloaded = keras.models.load_model(model_path, compile=False)
    probe_after = reloaded.predict(probe_inputs, verbose=0).reshape(-1)
    reload_verified = bool(np.allclose(probe_before, probe_after, rtol=2e-4, atol=2e-5))
    if not reload_verified:
        raise AssertionError("best_model.keras 재로드 prediction 검증 실패")
    details = {
        "final_epochs": int(final_epochs),
        "training_seconds": training_seconds,
        "model_file_size_bytes": int(model_path.stat().st_size),
        "model_file_size_mb": float(model_path.stat().st_size / 1024**2),
        "reload_prediction_verified": reload_verified,
    }
    return reloaded, details


def evaluate_official_validation_once(
    model: keras.Model,
    selected_model: str,
    selected_threshold: float,
    arrays: ExperimentArrays,
    config: ExperimentConfig,
    result_dir: Path,
) -> dict[str, Any]:
    print("\n=== 선택/epoch/threshold 고정 완료: 공식 validation 1회 평가 ===")
    validation_inputs = prepare_official_val_input(
        selected_model,
        arrays,
        config.normalization_mode,
    )
    started = time.perf_counter()
    probabilities = model.predict(validation_inputs, batch_size=256, verbose=0).reshape(-1)
    inference_seconds = float(time.perf_counter() - started)
    if not np.isfinite(probabilities).all():
        raise RuntimeError("공식 validation prediction에 NaN/Inf가 있습니다.")
    subject_frame = aggregate_subject_probabilities(
        arrays.y_val,
        probabilities,
        arrays.groups_val,
    )
    subject_metrics_05 = binary_metrics(
        subject_frame["label"], subject_frame["probability"], 0.5
    )
    subject_metrics_tuned = binary_metrics(
        subject_frame["label"], subject_frame["probability"], selected_threshold
    )
    window_metrics_05 = binary_metrics(arrays.y_val, probabilities, 0.5)
    window_metrics_tuned = binary_metrics(
        arrays.y_val,
        probabilities,
        selected_threshold,
    )
    window_frame = pd.DataFrame(
        {
            "model": selected_model,
            "window_index": np.arange(len(arrays.y_val)),
            "patient_id": arrays.groups_val,
            "window_start_date": arrays.start_val,
            "window_end_date": arrays.end_val,
            "label": arrays.y_val,
            "probability": probabilities,
            "pred_fixed_0_5": (probabilities >= 0.5).astype(int),
            "pred_oof_threshold": (probabilities >= selected_threshold).astype(int),
        }
    )
    subject_frame.insert(0, "model", selected_model)
    subject_frame["pred_fixed_0_5"] = (subject_frame["probability"] >= 0.5).astype(int)
    subject_frame["pred_oof_threshold"] = (
        subject_frame["probability"] >= selected_threshold
    ).astype(int)
    subject_frame["oof_threshold"] = selected_threshold
    window_frame.to_csv(result_dir / "official_val_window_predictions.csv", index=False)
    subject_frame.to_csv(result_dir / "official_val_subject_predictions.csv", index=False)
    print("공식 val subject @0.5:", json.dumps(json_ready(subject_metrics_05), ensure_ascii=False))
    print(
        "공식 val subject @OOF threshold:",
        json.dumps(json_ready(subject_metrics_tuned), ensure_ascii=False),
    )
    return {
        "probabilities": probabilities,
        "subject_frame": subject_frame,
        "window_frame": window_frame,
        "subject_metrics_05": subject_metrics_05,
        "subject_metrics_tuned": subject_metrics_tuned,
        "window_metrics_05_reference": window_metrics_05,
        "window_metrics_tuned_reference": window_metrics_tuned,
        "inference_seconds": inference_seconds,
    }


def _plot_confusion_matrix(ax: plt.Axes, matrix: Sequence[Sequence[int]], title: str) -> None:
    matrix_array = np.asarray(matrix)
    image = ax.imshow(matrix_array, cmap="Blues")
    for row in range(2):
        for column in range(2):
            ax.text(column, row, str(matrix_array[row, column]), ha="center", va="center")
    ax.set_xticks([0, 1], ["CN", "MCI+Dem"])
    ax.set_yticks([0, 1], ["CN", "MCI+Dem"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    ax.figure.colorbar(image, ax=ax, fraction=0.046, pad=0.04)


def save_evaluation_plots(
    selected_oof_subjects: pd.DataFrame,
    official_subjects: pd.DataFrame,
    selected_threshold: float,
    result_dir: Path,
) -> None:
    oof_y = selected_oof_subjects["label"].to_numpy(dtype=int)
    oof_p = selected_oof_subjects["probability"].to_numpy(dtype=float)
    val_y = official_subjects["label"].to_numpy(dtype=int)
    val_p = official_subjects["probability"].to_numpy(dtype=float)

    figure, axes = plt.subplots(2, 2, figsize=(10, 9))
    for axis, y_values, probabilities, threshold, title in (
        (axes[0, 0], oof_y, oof_p, 0.5, "Train OOF @0.5"),
        (axes[0, 1], oof_y, oof_p, selected_threshold, "Train OOF @selected"),
        (axes[1, 0], val_y, val_p, 0.5, "Official val @0.5"),
        (axes[1, 1], val_y, val_p, selected_threshold, "Official val @selected"),
    ):
        metrics = binary_metrics(y_values, probabilities, threshold)
        _plot_confusion_matrix(axis, metrics["confusion_matrix"], title)
    figure.tight_layout()
    figure.savefig(result_dir / "confusion_matrix.png", dpi=170, bbox_inches="tight")
    plt.close(figure)

    figure, axis = plt.subplots(figsize=(7, 6))
    for label, y_values, probabilities in (
        ("Train OOF", oof_y, oof_p),
        ("Official val", val_y, val_p),
    ):
        false_positive_rate, true_positive_rate, _ = roc_curve(y_values, probabilities)
        axis.plot(
            false_positive_rate,
            true_positive_rate,
            label=f"{label} AUC={safe_roc_auc(y_values, probabilities):.3f}",
        )
    axis.plot([0, 1], [0, 1], "--", color="gray")
    axis.set(xlabel="False positive rate", ylabel="True positive rate", title="Subject-level ROC")
    axis.legend()
    figure.tight_layout()
    figure.savefig(result_dir / "roc_curve.png", dpi=170, bbox_inches="tight")
    plt.close(figure)

    figure, axis = plt.subplots(figsize=(7, 6))
    for label, y_values, probabilities in (
        ("Train OOF", oof_y, oof_p),
        ("Official val", val_y, val_p),
    ):
        precision, recall, _ = precision_recall_curve(y_values, probabilities)
        axis.plot(
            recall,
            precision,
            label=f"{label} AP={average_precision_score(y_values, probabilities):.3f}",
        )
    axis.set(xlabel="Recall", ylabel="Precision", title="Subject-level precision-recall")
    axis.legend()
    figure.tight_layout()
    figure.savefig(result_dir / "pr_curve.png", dpi=170, bbox_inches="tight")
    plt.close(figure)

    figure, axis = plt.subplots(figsize=(7, 6))
    for label, y_values, probabilities in (
        ("Train OOF", oof_y, oof_p),
        ("Official val", val_y, val_p),
    ):
        fraction_positive, mean_predicted = calibration_curve(
            y_values,
            probabilities,
            n_bins=min(8, max(3, len(np.unique(probabilities)))),
            strategy="quantile",
        )
        axis.plot(mean_predicted, fraction_positive, marker="o", label=label)
    axis.plot([0, 1], [0, 1], "--", color="gray")
    axis.set(xlabel="Mean predicted probability", ylabel="Observed positive fraction", title="Subject-level calibration")
    axis.legend()
    figure.tight_layout()
    figure.savefig(result_dir / "calibration_curve.png", dpi=170, bbox_inches="tight")
    plt.close(figure)


def architecture_manifest() -> dict[str, Any]:
    return {
        MODEL_A: {
            "input": "X_integrated_seq fixed top-20, shape=(7,20)",
            "layers": [
                "Conv1D(32,kernel=3,same,L2=0.005)",
                "BatchNormalization",
                "ReLU",
                "Bidirectional LSTM(8,return_sequences=True,L2=0.005)",
                "GlobalAveragePooling1D",
                "Dense(8,relu,L2=0.005)",
                "Dropout(0.50)",
                "Dense(1,sigmoid,float32,L2=0.005)",
            ],
            "source": "06_datasanity_v2_subject_safe_lstm_(1).ipynb + final_model.keras",
        },
        MODEL_B: {
            "inputs": ["X_continuous_seq", "X_discrete_seq"],
            "continuous": [
                "padding -1 -> 0 + 5 observed masks",
                "TimeDistributed SeparableConv1D(16,5,same,swish)",
                "TimeDistributed Conv1D(32,12,stride=12,same,swish)",
                "reshape (7,24,32) -> (168,32)",
                "masked global average",
                "Dense(32,swish)+Dropout(0.30)",
            ],
            "discrete": [
                "LayerNormalization",
                "TimeDistributed Dense(16,swish)",
                "GlobalAveragePooling1D",
                "Dense(8,swish)+Dropout(0.20)",
            ],
            "fusion": "Concatenate -> Dense(16,swish) -> Dropout(0.35) -> sigmoid",
        },
        MODEL_C: {
            "base": f"same as {MODEL_B}",
            "transformer": {
                "tokens": 168,
                "d_model": 32,
                "blocks": 1,
                "num_heads": 2,
                "key_dim": 16,
                "ff_dim": 64,
                "dropout": 0.20,
                "pre_layer_normalization": True,
                "learnable_position_embedding": True,
                "attention_key_mask": "completely missing tokens excluded",
            },
        },
    }


def create_result_directory(root: Path) -> tuple[Path, str]:
    timestamp = datetime.now(SEOUL).strftime("%Y%m%d_%H%M%S")
    result_dir = Path(root) / timestamp
    result_dir.mkdir(parents=True, exist_ok=False)
    return result_dir, timestamp


def write_threshold_config(
    oof_results: dict[str, Any],
    selected_model: str,
    result_dir: Path,
) -> dict[str, Any]:
    thresholds = {
        model_name: float(result["threshold"])
        for model_name, result in oof_results["model_results"].items()
    }
    config = {
        "fixed_threshold": 0.5,
        "selection_source": "official train OOF subject predictions only",
        "criterion": "balanced_accuracy, tie=Youden_J, tie=closest_to_0.5",
        "aggregation": AGGREGATION_METHOD,
        "model_thresholds": thresholds,
        "selected_model": selected_model,
        "selected_threshold": thresholds[selected_model],
        "official_val_labels_used_for_threshold": False,
    }
    with (result_dir / "threshold_config.json").open("w", encoding="utf-8") as file:
        json.dump(json_ready(config), file, ensure_ascii=False, indent=2)
    return config


def write_run_manifest(
    *,
    config: ExperimentConfig,
    result_dir: Path,
    started_at: datetime,
    data_contract: dict[str, Any],
    normalization_details: dict[str, Any],
    tensorflow_info: dict[str, Any],
    mask_test: dict[str, Any],
    architectures: dict[str, dict[str, Any]],
    fold_manifest: pd.DataFrame,
    oof_results: dict[str, Any],
    threshold_config: dict[str, Any],
    final_refit: dict[str, Any],
    official_validation: dict[str, Any],
) -> dict[str, Any]:
    fold_subject_counts = (
        fold_manifest.groupby(["fold", "role", "label"])["patient_id"]
        .nunique()
        .rename("subjects")
        .reset_index()
        .to_dict("records")
    )
    selected_model = oof_results["selected_model"]
    manifest = {
        "run_started_at": started_at.isoformat(),
        "run_finished_at": datetime.now(SEOUL).isoformat(),
        "result_directory": str(result_dir.resolve()),
        "data_path": str(Path(config.data_path).resolve()),
        "source_dataset_reused_without_new_preprocessing_file": True,
        "reference_implementation": {
            "files_reviewed": [
                "build_lstm_dataset.py",
                "06_datasanity_v2_subject_safe_lstm_(1).ipynb",
                "final_model.keras",
                "generate_late_fusion_3class_notebook.py",
                "late_fusion_3class_colab_staged_ablation.ipynb",
            ],
            "baseline_checkpoint_input_shape": [7, 20],
            "baseline_checkpoint_total_parameters": 4_849,
            "baseline_checkpoint_trainable_parameters": 4_785,
            "stored_preprocessing": "train-only MinMax; official val transform only; continuous padding=-1.0",
            "baseline_extra_normalization": "fixed top-20 + per-subject z-score",
            "baseline_subject_aggregation": AGGREGATION_METHOD,
        },
        "pickle_keys": data_contract["keys"],
        "input_shapes": data_contract["input_shapes"],
        "class_mapping": CLASS_MAPPING,
        "data_contract": data_contract,
        "normalization": normalization_details,
        "padding_and_masking": {
            "padding_value": PADDING_VALUE,
            "padding_replacement": 0.0,
            "missingness_channels_added": 5,
            "token_validity": "any observed value in the corresponding 12-step downsample region",
            "attention": "invalid tokens excluded as keys",
            "pooling": "invalid tokens excluded; denominator clipped to >=1",
            "unit_test": mask_test,
        },
        "model_architecture": architecture_manifest(),
        "model_parameters": architectures,
        "training": {
            "new_model_optimizer": {
                "name": "Adam",
                "learning_rate": config.learning_rate,
                "clipnorm": config.clipnorm,
            },
            "baseline_optimizer": {
                "name": "Adam",
                "learning_rate": config.baseline_learning_rate,
                "clipnorm": None,
                "reason": "existing best LSTM training condition",
            },
            "loss": "BinaryCrossentropy",
            "batch_size": config.batch_size,
            "new_model_max_epochs": 3 if config.quick_mode else config.max_epochs,
            "baseline_max_epochs": 3 if config.quick_mode else config.baseline_max_epochs,
            "patience": 1 if config.quick_mode else config.patience,
            "early_stopping": (
                "subject-level ROC-AUC; exact tie broken by subject balanced accuracy @0.5; "
                "best weights restored"
            ),
            "reduce_lr": "val_loss, factor=0.5, min_lr=1e-5",
            "weighting": {
                "inverse_subject_window_count": config.inverse_subject_window_weight,
                "subject_level_class_weight": config.subject_class_weight,
                "formula": "subject_class_weight[label] / train_windows_for_subject, normalized to mean 1",
                "official_val_counts_used": False,
            },
        },
        "cross_validation": {
            "source_split": "split == train only",
            "method": "subject table StratifiedKFold expanded to all subject windows",
            "n_splits": 2 if config.quick_mode else config.n_splits,
            "seed": config.seed,
            "fold_subject_counts": fold_subject_counts,
            "same_folds_for_all_models": True,
        },
        "probability_aggregation": {
            "method": AGGREGATION_METHOD,
            "reason": "explicitly implemented by the existing best LSTM notebook",
            "primary_metrics_level": "subject",
            "window_metrics": "reference only",
        },
        "selection": {
            **oof_results["selection_details"],
            "priority": [
                "subject ROC-AUC",
                "subject PR-AUC",
                "subject balanced accuracy @0.5",
                "smaller fold ROC-AUC standard deviation",
            ],
            "official_val_used": False,
        },
        "threshold": threshold_config,
        "selected_model": selected_model,
        "final_refit": final_refit,
        "official_validation": {
            "evaluated_after_all_choices_frozen": True,
            "subject_metrics_at_0.5": official_validation["subject_metrics_05"],
            "subject_metrics_at_oof_threshold": official_validation[
                "subject_metrics_tuned"
            ],
            "window_metrics_at_0.5_reference": official_validation[
                "window_metrics_05_reference"
            ],
            "window_metrics_at_oof_threshold_reference": official_validation[
                "window_metrics_tuned_reference"
            ],
            "inference_seconds": official_validation["inference_seconds"],
        },
        "software": tensorflow_info,
        "quick_mode": config.quick_mode,
        "known_limitations": [
            "Overlapping 7-day windows are not independent observations.",
            "Only subject-level metrics are primary; window metrics are reference-only.",
            "match_baseline subject-history z-score assumes multiple windows are available for each subject at inference.",
            "The fixed baseline top-20 features were inherited from prior work rather than reselected inside these folds.",
            "Official validation is small (32 subjects in the verified pickle), so estimates have high uncertainty.",
            "The existing checkpoint notebook uses mean window probability aggregation, so mean-logit was not substituted.",
            "Model A keeps its original Adam learning rate/max-epoch settings; Models B/C use the requested new-model defaults.",
        ],
    }
    with (result_dir / "run_manifest.json").open("w", encoding="utf-8") as file:
        json.dump(json_ready(manifest), file, ensure_ascii=False, indent=2)
    return manifest


def required_output_files() -> list[str]:
    return [
        "model_summary.txt",
        "model_comparison.csv",
        "fold_metrics.csv",
        "fold_subject_manifest.csv",
        "oof_window_predictions.csv",
        "oof_subject_predictions.csv",
        "official_val_window_predictions.csv",
        "official_val_subject_predictions.csv",
        "best_model.keras",
        "run_manifest.json",
        "threshold_config.json",
        "confusion_matrix.png",
        "roc_curve.png",
        "pr_curve.png",
        "calibration_curve.png",
    ]


def verify_output_files(result_dir: Path) -> None:
    missing = [name for name in required_output_files() if not (result_dir / name).exists()]
    history_files = list(result_dir.glob("training_history_fold_*.csv"))
    if missing or not history_files:
        raise RuntimeError(
            f"필수 산출물 누락: files={missing}, training_history_count={len(history_files)}"
        )
    print("필수 산출물 검증 완료:", result_dir)


## Steps — 전체 실행 오케스트레이션

검증 → OOF → 선택/threshold 고정 → final refit → 공식 validation의 순서를 한 함수에 고정합니다.


In [9]:
def run_experiment(config: ExperimentConfig) -> Path:
    config = config.validated()
    started_at = datetime.now(SEOUL)
    tensorflow_info = configure_tensorflow(config)
    result_dir, _ = create_result_directory(config.result_root)
    print("RESULT_DIR:", result_dir.resolve())

    bundle = load_dataset(Path(config.data_path))
    data_contract = validate_data_contract(bundle)
    arrays = split_experiment_arrays(bundle, config)
    # split copy가 끝났으므로 원본 bundle 참조를 해제한다.
    del bundle
    gc.collect()

    folds, fold_manifest = build_subject_safe_folds(
        arrays.y_train,
        arrays.groups_train,
        config,
    )
    fold_manifest.to_csv(result_dir / "fold_subject_manifest.csv", index=False)
    prepared_by_model, normalization_details = prepare_training_inputs(
        arrays,
        config.normalization_mode,
    )
    mask_test = run_mask_unit_tests()
    architectures = inspect_model_architectures(
        arrays.X_disc_train.shape[-1],
        config,
        result_dir,
    )

    oof_results = run_oof_comparison(
        arrays,
        prepared_by_model,
        folds,
        architectures,
        config,
        result_dir,
    )
    selected_model = oof_results["selected_model"]
    threshold_config = write_threshold_config(oof_results, selected_model, result_dir)
    selected_threshold = float(threshold_config["selected_threshold"])
    selected_best_epochs = oof_results["model_results"][selected_model]["best_epochs"]
    final_epochs = max(1, int(round(float(np.median(selected_best_epochs)))))

    final_model, final_refit = fit_selected_final_model(
        selected_model,
        final_epochs,
        prepared_by_model[selected_model],
        arrays,
        config,
        result_dir,
    )
    official_validation = evaluate_official_validation_once(
        final_model,
        selected_model,
        selected_threshold,
        arrays,
        config,
        result_dir,
    )
    selected_oof_subjects = oof_results["model_results"][selected_model]["subject_frame"]
    save_evaluation_plots(
        selected_oof_subjects,
        official_validation["subject_frame"],
        selected_threshold,
        result_dir,
    )
    del final_model
    keras.backend.clear_session()
    gc.collect()

    write_run_manifest(
        config=config,
        result_dir=result_dir,
        started_at=started_at,
        data_contract=data_contract,
        normalization_details=normalization_details,
        tensorflow_info=tensorflow_info,
        mask_test=mask_test,
        architectures=architectures,
        fold_manifest=fold_manifest,
        oof_results=oof_results,
        threshold_config=threshold_config,
        final_refit=final_refit,
        official_validation=official_validation,
    )
    verify_output_files(result_dir)
    print("실험 완료:", result_dir.resolve())
    return result_dir


## Checks — mask unit test 후 전체 실험 실행

                아래 셀은 중간 사용자 입력 없이 끝까지 실행됩니다. `match_baseline`은 각 subject의 여러
                window를 함께 사용하는 오프라인 추론 조건이라는 한계가 manifest에 자동 기록됩니다.
                공식 validation label은 앞선 계약 검증 출력 외에는 선택/학습/threshold에 사용되지 않습니다.


In [10]:

experiment_config = ExperimentConfig(
    data_path=Path(DATA_PATH_OVERRIDE),
    result_root=Path(DATA_PATH_OVERRIDE).parent / "binary_conv_transformer_results",
    normalization_mode=NORMALIZATION_MODE,
    quick_mode=QUICK_MODE,
)
RESULT_DIR = run_experiment(experiment_config)
print("완료된 결과 폴더:", RESULT_DIR)


TensorFlow 설정: {"tensorflow_version": "2.20.0", "keras_version": "3.13.2", "gpu_devices": ["/physical_device:GPU:0"], "mixed_precision_policy": "mixed_float16"}
RESULT_DIR: /content/drive/MyDrive/GoogleAI_contest/Hyunsoo/previous/privious_lstm_preprocessed/binary_conv_transformer_results/20260721_161837
pickle 로드: /content/drive/MyDrive/GoogleAI_contest/Hyunsoo/previous/privious_lstm_preprocessed/lstm_dataset.pkl

=== 데이터 계약 검증 ===
shape: {'X_continuous_seq': [6940, 7, 288, 5], 'X_discrete_seq': [6940, 7, 47], 'X_integrated_seq': [6940, 7, 1487]}
NaN/Inf: {'X_continuous_seq': 0, 'X_discrete_seq': 0, 'X_integrated_seq': 0}
train {'windows': 5500, 'subjects': 138, 'window_label_counts': {0: 3359, 1: 2141}, 'subject_label_counts': {0: 84, 1: 54}}
val {'windows': 1440, 'subjects': 32, 'window_label_counts': {0: 1118, 1: 322}, 'subject_label_counts': {0: 26, 1: 6}}
continuous 변수별 padding 비율:
             feature  padding_ratio
 activity_class_5min       0.000658
   activity_met_5min       0

A_early_fusion_lstm: total_parameters=4,849, trainable_parameters=4,785


B_conv_only_intermediate: total_parameters=9,129, trainable_parameters=9,129


C_conv_transformer_intermediate: total_parameters=23,049, trainable_parameters=23,049

=== OOF: A_early_fusion_lstm ===
A_early_fusion_lstm fold=1: subject_auc=0.6973, subject_pr_auc=0.6637, best_epoch=9, seconds=36.6
A_early_fusion_lstm fold=2: subject_auc=0.6293, subject_pr_auc=0.5852, best_epoch=13, seconds=39.7
A_early_fusion_lstm fold=3: subject_auc=0.6484, subject_pr_auc=0.5537, best_epoch=20, seconds=49.7
A_early_fusion_lstm fold=4: subject_auc=0.6850, subject_pr_auc=0.6504, best_epoch=14, seconds=39.8

=== OOF: B_conv_only_intermediate ===
B_conv_only_intermediate fold=1: subject_auc=0.6395, subject_pr_auc=0.5375, best_epoch=15, seconds=61.8
B_conv_only_intermediate fold=2: subject_auc=0.6497, subject_pr_auc=0.6255, best_epoch=13, seconds=59.4
B_conv_only_intermediate fold=3: subject_auc=0.7363, subject_pr_auc=0.6379, best_epoch=43, seconds=121.7
B_conv_only_intermediate fold=4: subject_auc=0.6007, subject_pr_auc=0.5108, best_epoch=23, seconds=78.2

=== OOF: C_conv_transformer_

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'transformer_encoder_block_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


...

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'transformer_encoder_block_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


C_conv_transformer_intermediate fold=4: subject_auc=0.6557, subject_pr_auc=0.5293, best_epoch=1, seconds=45.9

=== Model comparison (subject-level primary) ===
                          model  subject_roc_auc  subject_pr_auc  subject_balanced_accuracy_at_05  subject_f1_at_05  subject_sensitivity_at_05  subject_specificity_at_05  subject_precision_at_05  subject_accuracy_at_05  subject_brier_score  oof_selected_threshold  subject_balanced_accuracy_at_oof_threshold  subject_f1_at_oof_threshold  subject_sensitivity_at_oof_threshold  subject_specificity_at_oof_threshold  subject_precision_at_oof_threshold  subject_accuracy_at_oof_threshold  window_roc_auc_reference  window_pr_auc_reference  window_balanced_accuracy_at_05_reference  window_balanced_accuracy_at_oof_threshold_reference  fold_subject_roc_auc_mean  fold_subject_roc_auc_std  fold_subject_pr_auc_mean  fold_subject_pr_auc_std  fold_subject_balanced_accuracy_mean  fold_subject_balanced_accuracy_std  fold_subject_f1_mean  fold_subje

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'transformer_encoder_block_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(



=== 선택/epoch/threshold 고정 완료: 공식 validation 1회 평가 ===


공식 val subject @0.5: {"roc_auc": 0.532051282051282, "pr_auc": 0.23811680469289165, "balanced_accuracy": 0.4807692307692308, "f1": 0.2608695652173913, "sensitivity": 0.5, "specificity": 0.46153846153846156, "precision": 0.17647058823529413, "accuracy": 0.46875, "brier_score": 0.2848601242568396, "threshold": 0.5, "tn": 12, "fp": 14, "fn": 3, "tp": 3, "confusion_matrix": [[12, 14], [3, 3]]}
공식 val subject @OOF threshold: {"roc_auc": 0.532051282051282, "pr_auc": 0.23811680469289165, "balanced_accuracy": 0.4807692307692308, "f1": 0.2608695652173913, "sensitivity": 0.5, "specificity": 0.46153846153846156, "precision": 0.17647058823529413, "accuracy": 0.46875, "brier_score": 0.2848601242568396, "threshold": 0.49720209263838255, "tn": 12, "fp": 14, "fn": 3, "tp": 3, "confusion_matrix": [[12, 14], [3, 3]]}
필수 산출물 검증 완료: /content/drive/MyDrive/GoogleAI_contest/Hyunsoo/previous/privious_lstm_preprocessed/binary_conv_transformer_results/20260721_161837
실험 완료: /content/drive/MyDrive/GoogleAI_conte

## Next Steps

                실행 후 `model_comparison.csv`에서 subject-level OOF 결과를 먼저 확인하고,
                `run_manifest.json`에서 선택 규칙·정규화·가중치·known limitations를 함께 검토하세요.
                공식 validation 지표는 모델과 threshold를 재조정하는 용도로 사용하지 않습니다.
